In [ ]:
import pandas as pd
from transformers import pipeline

df = pd.read_csv("stocktwits_sentiment_cleaned.csv")

model = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    return_all_scores=True
)

texts = df["cleaned_text"].fillna("").tolist()
results = model(texts, batch_size=16, truncation=True)

labels = []
confidences = []
positive_scores = []
negative_scores = []
neutral_scores = []

for result in results:
    scores = {item["label"].lower(): item["score"] for item in result}

    positive_scores.append(scores.get("positive", 0))
    negative_scores.append(scores.get("negative", 0))
    neutral_scores.append(scores.get("neutral", 0))

    label = max(scores, key=scores.get)
    labels.append(label.capitalize())
    confidences.append(scores[label])

df["finbert_positive"] = positive_scores
df["finbert_negative"] = negative_scores
df["finbert_neutral"] = neutral_scores
df["finbert_sentiment"] = labels
df["finbert_confidence"] = confidences

df["finbert_score"] = df["finbert_positive"] - df["finbert_negative"]

print("Sentiment distribution:")
print(df["finbert_sentiment"].value_counts())

print("\nAverage FinBERT score:", round(df["finbert_score"].mean(), 4))

print(df[[
    "ticker",
    "cleaned_text",
    "finbert_sentiment",
    "finbert_confidence",
    "finbert_score"
]].head())

df.to_csv("stocktwits_finbert_results.csv", index=False)
print("\nFinBERT results saved.")